In [20]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'green-trips'

In [21]:
from ride_models import GreenRide

In [22]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=GreenRide.ride_deserializer
)

In [23]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [24]:
cur.execute("""
DROP TABLE IF EXISTS green_processed_events
""")

In [25]:
cur.execute("""
CREATE TABLE IF NOT EXISTS green_processed_events (
    pickup_datetime TIMESTAMP,
    dropoff_datetime TIMESTAMP,
    PULocationID INTEGER,
    DOLocationID INTEGER,
    passenger_count INTEGER,
    trip_distance DOUBLE PRECISION,
    tip_amount DOUBLE PRECISION,
    total_amount DOUBLE PRECISION
)
""")

In [26]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
count_condition = 0
try:

    for message in consumer:
        ride = message.value
        pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
        dropoff_dt = datetime.fromtimestamp(ride.lpep_dropoff_datetime / 1000)
        cur.execute(
            """INSERT INTO green_processed_events (
                pickup_datetime,
                dropoff_datetime,
                PULocationID,
                DOLocationID,
                passenger_count,
                trip_distance,
                tip_amount,
                total_amount
            )
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)""",
            (pickup_dt, dropoff_dt,
            ride.PULocationID, ride.DOLocationID,
            ride.passenger_count, ride.trip_distance, 
            ride.tip_amount, ride.total_amount)
        )
        count += 1
        if count % 100 == 0:
            print()
            print(f"Inserted {count} rows.")
        else:
            print(".", end="")

        if ride.trip_distance > 5:
            count_condition += 1


except KeyboardInterrupt as ki:
    print(ki)

print("number of rides which trip_distance > 5: ", count_condition)


consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...
...................................................................................................
Inserted 100 rows.
...................................................................................................
Inserted 200 rows.
...................................................................................................
Inserted 300 rows.
...................................................................................................
Inserted 400 rows.
...................................................................................................
Inserted 500 rows.
...................................................................................................
Inserted 600 rows.
...................................................................................................
Inserted 700 rows.
...................................................................................................
Inserted 800 